# CVEDataTabulator: Uses CPE id as input to fetch device-specific CVE

In [ ]:
import os
import json
import re
import shutil
import time

import requests
import pandas as pd

from urllib.parse import quote

from openpyxl import Workbook
from openpyxl.styles import (
    Font,
    PatternFill,
    Alignment,
    Border,
    Side,
    GradientFill
)
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule, DataBarRule

In [ ]:

def fetch_cves_by_cpe(cpe_name, results_per_page=2000, delay=6):
    """
    Fetch CVEs from NVD API based on CPE name
    """
    base_url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    encoded_cpe = quote(cpe_name, safe='')

    all_vulnerabilities = []
    start_index = 0

    print(f"Fetching CVEs for CPE: {cpe_name}")
    print(f"Using delay of {delay} seconds between requests...")

    while True:
        url = f"{base_url}?cpeName={encoded_cpe}&resultsPerPage={results_per_page}&startIndex={start_index}"
        print(f"Fetching results starting from index {start_index}...")

        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()

            data = response.json()
            vulnerabilities = data.get('vulnerabilities', [])
            total_results = data.get('totalResults', 0)

            print(f"Retrieved {len(vulnerabilities)} CVEs (Total: {total_results})")

            if not vulnerabilities:
                break

            all_vulnerabilities.extend(vulnerabilities)

            if len(all_vulnerabilities) >= total_results:
                break

            start_index += results_per_page

            print(f"Waiting {delay} seconds before next request...")
            time.sleep(delay)

        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
            break
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON response: {e}")
            break

    print(f"Total CVEs fetched: {len(all_vulnerabilities)}")
    return all_vulnerabilities


def process_and_save_cves(cpe_name):
    """
    Fetch CVEs by CPE and organize them into accepted/rejected
    """
    cve_dir = "cve_data"
    rejected_dir = "rejected_cve_data"

    os.makedirs(cve_dir, exist_ok=True)
    os.makedirs(rejected_dir, exist_ok=True)

    vulnerabilities = fetch_cves_by_cpe(cpe_name)

    if not vulnerabilities:
        print("No CVEs found for the specified CPE.")
        return

    total_cves = len(vulnerabilities)
    rejected_count = 0
    accepted_count = 0

    print("\nProcessing and saving CVEs...")

    for vuln in vulnerabilities:
        cve_data = vuln.get('cve', {})
        cve_id = cve_data.get('id', 'UNKNOWN')
        vuln_status = cve_data.get('vulnStatus', '')

        filename = f"{cve_id}.json"

        if vuln_status == "Rejected":
            file_path = os.path.join(rejected_dir, filename)
            rejected_count += 1
        else:
            file_path = os.path.join(cve_dir, filename)
            accepted_count += 1

        try:
            with open(file_path, "w", encoding="utf-8") as f:
                json.dump(cve_data, f, indent=2, ensure_ascii=False)
        except Exception as e:
            print(f"Error saving {cve_id}: {e}")

    print(f"\nPROCESSING COMPLETE")
    print(f"Total CVEs processed: {total_cves}")
    print(f"Accepted CVEs saved to '{cve_dir}': {accepted_count}")
    print(f"Rejected CVEs saved to '{rejected_dir}': {rejected_count}")

    return {'total': total_cves, 'accepted': accepted_count, 'rejected': rejected_count}


# CPE for the specific device you want CVEs for (change this per device)
#Eample
cpe_name = "cpe:2.3:h:baxter:sigma_spectrum_infusion_system:-:*:*:*:*:*:*:*"

print("CVE Fetcher by CPE")
print(" " * 50)

try:
    results = process_and_save_cves(cpe_name)
    print(f"\nFiles can be found in:")
    print(f"  - Accepted CVEs: ./cve_data/")
    print(f"  - Rejected CVEs: ./rejected_cve_data/")
except KeyboardInterrupt:
    print("\nOperation cancelled by user.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:


cve_dir = "cve_data"
rejected_dir = "rejected_cve_data"

os.makedirs(rejected_dir, exist_ok=True)

removed_count = 0
total_files = 0

for filename in os.listdir(cve_dir):
    if filename.endswith(".json"):
        total_files += 1
        file_path = os.path.join(cve_dir, filename)

        with open(file_path, "r", encoding="utf-8") as f:
            cve_entry = json.load(f)

        if cve_entry.get("vulnStatus") == "Rejected":
            removed_count += 1
            shutil.move(file_path, os.path.join(rejected_dir, filename))

print(f"Total files processed: {total_files}")
print(f"Rejected CVEs moved: {removed_count}")
print(f"Remaining CVEs in original folder: {total_files - removed_count}")

In [ ]:
# Shared CPE / CVSS parsing helpers
def parse_cpe(cpe_str):
    """Parse CPE 2.3 URI into (vendor, product, version)."""
    parts = cpe_str.split(":")
    vendor  = parts[3] if len(parts) > 3 else "N/A"
    product = parts[4] if len(parts) > 4 else "N/A"
    version = parts[5] if len(parts) > 5 else "N/A"
    return vendor.replace("_", " ").title(), product.replace("_", " ").title(), version

def extract_cvss(metrics):
    """
    Return a normalized dict of CVSS fields from a CVE's 'metrics' block,
    preferring v3.1, then v3.0, then v2.0.
    """
    m = {}
    if "cvssMetricV31" in metrics:
        raw = metrics["cvssMetricV31"][0]
        d = raw["cvssData"]
        m.update({
            "CVSS Version": "3.1",
            "baseScore": d.get("baseScore"), "baseSeverity": d.get("baseSeverity"),
            "attackVector": d.get("attackVector"), "attackComplexity": d.get("attackComplexity"),
            "privilegesRequired": d.get("privilegesRequired"), "userInteraction": d.get("userInteraction"),
            "scope": d.get("scope"), "confidentialityImpact": d.get("confidentialityImpact"),
            "integrityImpact": d.get("integrityImpact"), "availabilityImpact": d.get("availabilityImpact"),
            "exploitabilityScore": raw.get("exploitabilityScore"), "impactScore": raw.get("impactScore"),
            "vectorString": d.get("vectorString"),
        })
    elif "cvssMetricV30" in metrics:
        raw = metrics["cvssMetricV30"][0]
        d = raw["cvssData"]
        m.update({
            "CVSS Version": "3.0",
            "baseScore": d.get("baseScore"), "baseSeverity": d.get("baseSeverity"),
            "attackVector": d.get("attackVector"), "attackComplexity": d.get("attackComplexity"),
            "privilegesRequired": d.get("privilegesRequired"), "userInteraction": d.get("userInteraction"),
            "scope": d.get("scope"), "confidentialityImpact": d.get("confidentialityImpact"),
            "integrityImpact": d.get("integrityImpact"), "availabilityImpact": d.get("availabilityImpact"),
            "exploitabilityScore": raw.get("exploitabilityScore"), "impactScore": raw.get("impactScore"),
            "vectorString": d.get("vectorString"),
        })
    elif "cvssMetricV2" in metrics:
        raw = metrics["cvssMetricV2"][0]
        d = raw["cvssData"]
        m.update({
            "CVSS Version": "2.0",
            "baseScore": d.get("baseScore"), "baseSeverity": raw.get("baseSeverity"),
            "attackVector": d.get("accessVector"), "attackComplexity": d.get("accessComplexity"),
            "privilegesRequired": d.get("authentication"),
            "userInteraction": "REQUIRED" if raw.get("userInteractionRequired") else "NONE",
            "scope": "N/A",
            "confidentialityImpact": d.get("confidentialityImpact"),
            "integrityImpact": d.get("integrityImpact"),
            "availabilityImpact": d.get("availabilityImpact"),
            "exploitabilityScore": raw.get("exploitabilityScore"), "impactScore": raw.get("impactScore"),
            "vectorString": d.get("vectorString"),
        })
    return m

In [ ]:


json_dir = "cve_data"



#  parse all files 
main_rows, config_rows, ref_rows = [], [], []

for fname in sorted(os.listdir(json_dir)):
    if not fname.endswith(".json"):
        continue
    data = json.load(open(os.path.join(json_dir, fname), encoding="utf-8"))
    cve_id = data["id"]

    desc = next((d["value"] for d in data.get("descriptions", []) if d.get("lang") == "en"), "N/A")

    weak_primary = weak_secondary = "N/A"
    for w in data.get("weaknesses", []):
        val = next((d["value"] for d in w.get("description", []) if d.get("lang") == "en"), None)
        if val:
            if w.get("type") == "Primary":
                weak_primary = val
            else:
                weak_secondary = val

    cvss = extract_cvss(data.get("metrics", {}))

    main_rows.append({
        "CVE ID": cve_id,
        "Published": data.get("published", "")[:10],
        "Last Modified": data.get("lastModified", "")[:10],
        "Status": data.get("vulnStatus", "N/A"),
        "Source": data.get("sourceIdentifier", "N/A"),
        "Description": desc,
        "Primary Weakness (CWE)": weak_primary,
        "Secondary Weakness (CWE)": weak_secondary,
        "CVSS Version": cvss.get("CVSS Version"),
        "Base Score": cvss.get("baseScore"),
        "Severity": cvss.get("baseSeverity"),
        "Attack Vector": cvss.get("attackVector"),
        "Attack Complexity": cvss.get("attackComplexity"),
        "Privileges Required": cvss.get("privilegesRequired"),
        "User Interaction": cvss.get("userInteraction"),
        "Scope": cvss.get("scope"),
        "Confidentiality Impact": cvss.get("confidentialityImpact"),
        "Integrity Impact": cvss.get("integrityImpact"),
        "Availability Impact": cvss.get("availabilityImpact"),
        "Exploitability Score": cvss.get("exploitabilityScore"),
        "Impact Score": cvss.get("impactScore"),
        "Vector String": cvss.get("vectorString"),
    })

    for cfg in data.get("configurations", []):
        for node in cfg.get("nodes", []):
            for cpe in node.get("cpeMatch", []):
                vendor, product, version = parse_cpe(cpe.get("criteria", ""))
                config_rows.append({
                    "CVE ID": cve_id,
                    "Vulnerable": cpe.get("vulnerable", False),
                    "Vendor": vendor,
                    "Product / Component": product,
                    "Version": version,
                    "CPE Criteria": cpe.get("criteria", "N/A"),
                })

    seen_urls = set()
    for ref in data.get("references", []):
        url = ref.get("url", "")
        if url in seen_urls:
            continue
        seen_urls.add(url)
        ref_rows.append({
            "CVE ID": cve_id,
            "URL": url,
            "Tags": ", ".join(ref.get("tags", [])),
        })

df_main   = pd.DataFrame(main_rows)
df_config = pd.DataFrame(config_rows)
df_ref    = pd.DataFrame(ref_rows)

#  build Excel 
wb = Workbook()

C_HEADER_BG  = "1A2B4A"
C_HEADER_FG  = "FFFFFF"
C_ACCENT     = "E63946"
C_ROW_ALT    = "F0F4F8"
C_BORDER     = "C8D0DC"

SEVERITY_COLORS = {
    "CRITICAL": "7B0000",
    "HIGH":     "C0392B",
    "MEDIUM":   "D4860A",
    "LOW":      "2E7D32",
    "NONE":     "558B2F",
}
SEV_FG = {k: "FFFFFF" for k in SEVERITY_COLORS}
SEV_FG["LOW"] = "FFFFFF"

thin = Side(border_style="thin", color=C_BORDER)
border = Border(left=thin, right=thin, top=thin, bottom=thin)

def style_header(cell, bg=C_HEADER_BG, fg=C_HEADER_FG):
    cell.font = Font(bold=True, color=fg, name="Calibri", size=10)
    cell.fill = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell.border = border

def style_data(cell, alt=False):
    cell.font = Font(name="Calibri", size=10)
    if alt:
        cell.fill = PatternFill("solid", fgColor=C_ROW_ALT)
    cell.alignment = Alignment(vertical="top", wrap_text=True)
    cell.border = border

def autofit(ws, min_w=10, max_w=50):
    for col in ws.columns:
        length = max((len(str(c.value or "")) for c in col), default=0)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max(length + 2, min_w), max_w)

def write_sheet(ws, df, col_widths=None):
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    headers = list(df.columns)
    for ci, h in enumerate(headers, 1):
        cell = ws.cell(row=1, column=ci, value=h)
        style_header(cell)
    for ri, row in enumerate(df.itertuples(index=False), 2):
        alt = ri % 2 == 0
        for ci, val in enumerate(row, 1):
            cell = ws.cell(row=ri, column=ci, value=val)
            style_data(cell, alt)
    ws.row_dimensions[1].height = 28
    if col_widths:
        for col, w in col_widths.items():
            ws.column_dimensions[col].width = w
    else:
        autofit(ws)

#  Sheet 1: CVE Summary 
ws1 = wb.active
ws1.title = "CVE Summary"
write_sheet(ws1, df_main, col_widths={
    "A": 18, "B": 12, "C": 14, "D": 14, "E": 28, "F": 60,
    "G": 20, "H": 22, "I": 10, "J": 10, "K": 10, "L": 15,
    "M": 16, "N": 18, "O": 16, "P": 10, "Q": 18, "R": 16,
    "S": 18, "T": 16, "U": 16, "V": 35,
})

sev_col_idx = df_main.columns.get_loc("Severity") + 1
for ri in range(2, len(df_main) + 2):
    cell = ws1.cell(row=ri, column=sev_col_idx)
    sev = str(cell.value or "").upper()
    if sev in SEVERITY_COLORS:
        cell.fill = PatternFill("solid", fgColor=SEVERITY_COLORS[sev])
        cell.font = Font(bold=True, color="FFFFFF", name="Calibri", size=10)

score_col = get_column_letter(df_main.columns.get_loc("Base Score") + 1)
score_range = f"{score_col}2:{score_col}{len(df_main)+1}"
ws1.conditional_formatting.add(score_range,
    ColorScaleRule(start_type="num", start_value=0, start_color="63BE7B",
                   mid_type="num",   mid_value=5,   mid_color="FFEB84",
                   end_type="num",   end_value=10,  end_color="F8696B"))

#  Sheet 2: Affected Devices / Components 
ws2 = wb.create_sheet("Affected Components")
write_sheet(ws2, df_config, col_widths={"A": 18, "B": 12, "C": 20, "D": 30, "E": 12, "F": 55})

for ri in range(2, len(df_config) + 2):
    cell = ws2.cell(row=ri, column=2)  # "Vulnerable"
    if cell.value is True:
        cell.value = "YES"
        cell.fill = PatternFill("solid", fgColor="FDECEA")
        cell.font = Font(bold=True, color="C0392B", name="Calibri", size=10)
    else:
        cell.value = "no"

#  Sheet 3: References 
ws3 = wb.create_sheet("References")
write_sheet(ws3, df_ref, col_widths={"A": 18, "B": 65, "C": 40})

#  Sheet 4: Analytics Summary 
ws4 = wb.create_sheet("Analytics")
ws4.sheet_view.showGridLines = False

def h1(ws, row, col, text):
    c = ws.cell(row=row, column=col, value=text)
    c.font = Font(bold=True, color=C_HEADER_FG, name="Calibri", size=12)
    c.fill = PatternFill("solid", fgColor=C_HEADER_BG)
    c.alignment = Alignment(horizontal="left", vertical="center")
    c.border = border
    return c

def kv(ws, row, lbl, val, col=1):
    lc = ws.cell(row=row, column=col, value=lbl)
    lc.font = Font(bold=True, name="Calibri", size=10)
    lc.fill = PatternFill("solid", fgColor="E8EDF3")
    lc.border = border
    lc.alignment = Alignment(horizontal="left", vertical="center")
    vc = ws.cell(row=row, column=col+1, value=val)
    vc.font = Font(name="Calibri", size=10)
    vc.border = border
    vc.alignment = Alignment(horizontal="center", vertical="center")
    return vc

h1(ws4, 1, 1, "📊  CVE Dataset — Analytics Overview")
ws4.merge_cells("A1:F1")
ws4.row_dimensions[1].height = 30

kv(ws4, 3, "Total CVEs", len(df_main))
kv(ws4, 4, "Total Affected Components", len(df_config[df_config["Vulnerable"] == True]))
kv(ws4, 5, "Total References", len(df_ref))
kv(ws4, 6, "Date Range (Published)",
   f"{df_main['Published'].min()}  →  {df_main['Published'].max()}")

sev_counts = df_main["Severity"].value_counts()
h1(ws4, 9, 1, "Severity Breakdown")
ws4.merge_cells("A9:B9")
for i, (sev, cnt) in enumerate(sev_counts.items(), 10):
    c_lbl = ws4.cell(row=i, column=1, value=sev)
    c_cnt = ws4.cell(row=i, column=2, value=cnt)
    bg = SEVERITY_COLORS.get(str(sev).upper(), "888888")
    for c in (c_lbl, c_cnt):
        c.fill = PatternFill("solid", fgColor=bg)
        c.font = Font(bold=True, color="FFFFFF", name="Calibri", size=10)
        c.border = border
        c.alignment = Alignment(horizontal="center", vertical="center")

av_counts = df_main["Attack Vector"].value_counts()
h1(ws4, 9, 4, "Attack Vector Breakdown")
ws4.merge_cells("D9:E9")
for i, (av, cnt) in enumerate(av_counts.items(), 10):
    c_lbl = ws4.cell(row=i, column=4, value=av)
    c_cnt = ws4.cell(row=i, column=5, value=cnt)
    for c in (c_lbl, c_cnt):
        c.fill = PatternFill("solid", fgColor="2C3E50")
        c.font = Font(color="FFFFFF", name="Calibri", size=10, bold=True)
        c.border = border
        c.alignment = Alignment(horizontal="center", vertical="center")

vendor_counts = df_config[df_config["Vulnerable"] == True]["Vendor"].value_counts().head(10)
h1(ws4, 20, 1, "Top Affected Vendors (vulnerable components)")
ws4.merge_cells("A20:C20")
for i, (v, cnt) in enumerate(vendor_counts.items(), 21):
    c_lbl = ws4.cell(row=i, column=1, value=v)
    c_cnt = ws4.cell(row=i, column=2, value=cnt)
    c_lbl.font = Font(name="Calibri", size=10)
    c_cnt.font = Font(name="Calibri", size=10, bold=True)
    c_cnt.fill = PatternFill("solid", fgColor="DDEEFF")
    for c in (c_lbl, c_cnt):
        c.border = border
        c.alignment = Alignment(horizontal="left", vertical="center")

h1(ws4, 20, 4, "Score Statistics")
ws4.merge_cells("D20:E20")
scores = df_main["Base Score"].dropna()
for i, (lbl, val) in enumerate([
    ("Average Base Score", round(scores.mean(), 2)),
    ("Max Base Score",     scores.max()),
    ("Min Base Score",     scores.min()),
    ("Median Base Score",  scores.median()),
], 21):
    ws4.cell(row=i, column=4, value=lbl).border = border
    ws4.cell(row=i, column=4).font = Font(name="Calibri", size=10)
    ws4.cell(row=i, column=4).alignment = Alignment(horizontal="left")
    c = ws4.cell(row=i, column=5, value=val)
    c.border = border
    c.font = Font(bold=True, name="Calibri", size=10)
    c.alignment = Alignment(horizontal="center")

for col, w in [("A",28),("B",12),("C",12),("D",32),("E",12)]:
    ws4.column_dimensions[col].width = w

#  save 
out_path = "cve_data_formated.xlsx"
wb.save(out_path)
print("Saved:", out_path)
print("Main rows:", len(df_main))
print("Config rows:", len(df_config))
print("Ref rows:", len(df_ref))

summary = {
    "cves": df_main.to_dict(orient="records"),
    "components": df_config.assign(Vulnerable=df_config["Vulnerable"].astype(str)).to_dict(orient="records"),
    "references": df_ref.to_dict(orient="records"),
}
# json.dump(summary, open("cve_summary.json", "w"), indent=2)
print("JSON summary saved.")

# Tabulating the CVEs entries in an excel sheet for Data Curation

In [ ]:
# Define the directory containing JSON files
json_dir = "cve_data"

# List to store extracted CVE data
cve_data = []

# Iterate over all JSON files in the directory
for filename in os.listdir(json_dir):
    if filename.endswith(".json"):  # Process only JSON files
        file_path = os.path.join(json_dir, filename)

        # Open and read the JSON file
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)

        # Extract the CVE ID
        cve_id = data.get("id")

        # Extract English description
        description = "N/A"
        descriptions = data.get("descriptions", [])
        for desc in descriptions:
            if desc.get("lang") == "en":
                description = desc.get("value", "N/A")
                break

        # Extract weaknesses
        weaknesses = []
        for weakness in data.get("weaknesses", []):
            for desc in weakness.get("description", []):
                if desc.get("lang") == "en":
                    weaknesses.append(desc.get("value", ""))
                    break

        # Join multiple weaknesses with a semicolon if present
        weakness_string = "; ".join(weaknesses) if weaknesses else "N/A"

        # Initialize metrics with default values
        metrics = {
            "CVE ID": cve_id,
            "Description": description,
            "Weaknesses": weakness_string,
            "baseScore": None,
            "baseSeverity": None,
            "attackVector": None,
            "attackComplexity": None,
            "privilegesRequired": None,
            "userInteraction": None,
            "scope": None,
            "confidentialityImpact": None,
            "integrityImpact": None,
            "availabilityImpact": None,
            "exploitabilityScore": None,
            "CVSS Version": None
        }

        cvss = extract_cvss(data.get("metrics", {}))
        metrics["baseScore"] = cvss.get("baseScore")
        metrics["baseSeverity"] = cvss.get("baseSeverity")
        metrics["attackVector"] = cvss.get("attackVector")
        metrics["attackComplexity"] = cvss.get("attackComplexity")
        metrics["privilegesRequired"] = cvss.get("privilegesRequired")
        metrics["userInteraction"] = cvss.get("userInteraction")
        metrics["scope"] = cvss.get("scope")
        metrics["confidentialityImpact"] = cvss.get("confidentialityImpact")
        metrics["integrityImpact"] = cvss.get("integrityImpact")
        metrics["availabilityImpact"] = cvss.get("availabilityImpact")
        metrics["exploitabilityScore"] = cvss.get("exploitabilityScore")
        metrics["CVSS Version"] = cvss.get("CVSS Version")

        cve_data.append(metrics)
# Create a DataFrame from the extracted data
df = pd.DataFrame(cve_data)
df

In [ ]:
df = df.drop(columns=["scope"])
nan_rows_df = df[df.isna().any(axis=1)]  # Select rows where any column has NaN
nan_rows_df
df["Weaknesses"] = df["Weaknesses"].replace(["N/A","<NA>",""], pd.NA)
na_count = df["Weaknesses"].isna().sum()
print(na_count)

In [ ]:
def clean_cwe_column(df):
    df['Weaknesses'] = df['Weaknesses'].astype(str)  # Ensure all values are strings

    # Split, remove duplicates, and filter out unwanted entries
    def process_weaknesses(entry):
        items = set(entry.split('; '))
        items.discard('NVD-CWE-noinfo')
        items.discard('NVD-CWE-Other')
        return '; '.join(sorted(items)) if items else ''

    df['Weaknesses'] = df['Weaknesses'].apply(process_weaknesses)
    return df


# Apply the function
df = clean_cwe_column(df)

In [ ]:
# Store the total number of rows before filtering
total_rows_before = df.shape[0]

# Remove rows where 'Weaknesses' is NaN
df_filtered = df.dropna(subset=["Weaknesses"])

# Store the total number of rows after filtering
total_rows_after = df_filtered.shape[0]

# Calculate and print the number of rows removed
rows_removed = total_rows_before - total_rows_after
print(f"Total rows removed: {rows_removed}")

In [ ]:
# Save to an Excel file
output_file = "cve_data.xlsx"
df_filtered.to_excel(output_file, index=False)

print(f"Extracted {len(cve_data)} CVE entries with metrics and saved to {output_file}.")

In [ ]:
df